# 72 — Agreement with lme4 and mirt (paper Section 7.2)

irtree is compared with two maximum-likelihood programs on models both can fit, using the verbal aggression data.

* **Part 1, lme4** (`glmer`, Laplace approximation): the logistic Rasch linear tree, (A) with item-by-node locations and (B) explanatory, with random items and covariates.
* **Part 2, mirt** (marginal ML with quadrature): (M1) the linear tree with logistic 2PL nodes and correlated traits; (M2) a one-node generalized partial credit model.

**Run the R scripts first** (from this folder): `72a_lme4_reference.R` for Part 1 and `72b_mirt_reference.R` for Part 2. They write their estimates to `results/`.

**What agreement means here.** With weak priors and 316 × 24 responses, posterior means should be close to the ML estimates, and posterior SDs close to the standard errors. Two systematic differences are expected and are not errors: (i) a variance component estimated from few units (the 24 item locations of a node) is pulled by its prior and is right-skewed, so its posterior mean exceeds its ML estimate; (ii) glmer's standard errors of fixed effects treat the estimated variance components as known, so they are smaller than posterior SDs that integrate over them.

## Setup

In [5]:
# Install the package from this repository (editable), once per environment.
# The notebook lives in replication/<folder>/, so the repository root is two levels up.
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1]
!{sys.executable} -m pip install -q -e "C:\Users\mehoujm\Documents\GitHub\irtree-0.4.0"

In [6]:
import warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import irtree
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore", category=FutureWarning)

RESULTS = pathlib.Path("results"); RESULTS.mkdir(exist_ok=True)

# Runtime log: wrap every expensive step in `with timed("label"):`.
# The last cell prints the table and saves it to results/.
import time, contextlib
RUNTIME = []

@contextlib.contextmanager
def timed(step):
    t0 = time.perf_counter()
    yield
    dt = time.perf_counter() - t0
    RUNTIME.append({"step": step, "seconds": round(dt, 1)})
    print(f"[{step}] " + (f"{dt / 60:.1f} min" if dt >= 60 else f"{dt:.1f} s"))

ModuleNotFoundError: No module named 'irtree'

In [ ]:
N_ITER   = 4000
N_CHAINS = 4
SEED     = 2026
VAGUE    = irtree.NormalPrior(mean=0.0, variance=10.0)   # stands in for "no prior" (ML fixed effects)

## Step 1 — Data, covariates and the tree
The same coding as `72a_lme4_reference.R` and notebook 61: centered covariates and an item `intercept` column whose coefficient is the node intercept.

In [ ]:
va = pd.read_csv("verbagg_long.csv")
va["response"] = va["resp"].map({"no": 0, "perhaps": 1, "yes": 2})
ITEMS = list(dict.fromkeys(va["item"]))          # lme4's item order

def centered(x):
    x = x.astype(float)
    return x - x.mean()

pers, items = va.drop_duplicates("id"), va.drop_duplicates("item")
person_cov = pd.DataFrame({"id": pers["id"], "anger": centered(pers["Anger"]),
                           "male": centered(pers["Gender"] == "M")})
item_cov = pd.DataFrame({"item": items["item"], "intercept": 1.0,
                         "scold": centered(items["btype"] == "scold"),
                         "shout": centered(items["btype"] == "shout"),
                         "self":  centered(items["situ"] == "self"),
                         "do":    centered(items["mode"] == "do")})
table = va[["id", "item", "response"]].merge(person_cov, on="id").merge(item_cov, on="item")
X_NAMES, Q_NAMES = ["anger", "male"], ["intercept", "scold", "shout", "self", "do"]

linear_tree = irtree.IRTreeSpec(mapping=np.array([[0, np.nan], [1, 0], [1, 1]]))
data_plain = irtree.load_data(table[["id", "item", "response"]], spec=linear_tree, person_id="id",
                              item_id="item", response_col="response", categories_start=0)
data_cov = irtree.load_data(table, spec=linear_tree, person_id="id", item_id="item",
                            response_col="response", roles={"X": X_NAMES, "Q": Q_NAMES},
                            categories_start=0)
assert list(data_plain.item_ids) == ITEMS

## Step 2 — Helpers
`compare` lines up irtree posterior means and SDs with the reference estimates and standard errors, and reports the largest difference in units of the reference SE. `draws` returns the pooled posterior draws of one parameter (used for derived quantities such as d = −a·b).

In [ ]:
def compare(rows, label):
    df = pd.DataFrame(rows, columns=["parameter", "irtree_mean", "irtree_sd", "ref_est", "ref_se"])
    df["diff"] = df["irtree_mean"] - df["ref_est"]
    df["diff_in_se"] = df["diff"] / df["ref_se"]
    print(f"{label}: {len(df)} parameters, max |diff| = {df['diff'].abs().max():.3f}, "
          f"max |diff|/SE = {df['diff_in_se'].abs().max():.2f}, "
          f"corr = {np.corrcoef(df['irtree_mean'], df['ref_est'])[0, 1]:.4f}")
    return df

def draws(fit, name):
    return np.concatenate([r.chain[:, r.param_names.index(name)].astype(float) for r in fit.results])

def scatter(df, title, ax):
    lo, hi = df[["irtree_mean", "ref_est"]].min().min(), df[["irtree_mean", "ref_est"]].max().max()
    ax.plot([lo, hi], [lo, hi], color="0.7", lw=1)
    ax.errorbar(df["ref_est"], df["irtree_mean"], yerr=df["irtree_sd"], fmt="o", ms=3, lw=0.8)
    ax.set_xlabel("reference estimate"); ax.set_ylabel("irtree posterior mean"); ax.set_title(title)

# Part 1 — lme4
## Step 3 — Model A: item-by-node locations
Logistic Rasch nodes; the item locations get a nearly flat N(0, 10) prior so they behave like glmer's fixed effects. glmer's coefficient of `item:node` is −b. The trait correlation has the default LKJ(1) prior; the variances (Rasch dimensions) the default InvGamma(1, 1).

In [ ]:
lme4_A = pd.read_csv(RESULTS / "lme4_modelA.csv").set_index("term")
nodes_A = [irtree.LogitRaschNode(), irtree.LogitRaschNode()]
priors_A = irtree.build_default_priors(linear_tree, nodes_A)
priors_A.set("b", VAGUE)
with timed("irtree Model A"):
    fit_A = irtree.irtree_fit(data_plain, linear_tree, nodes_A, priors=priors_A,
                              n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED, progress=False)
sA = fit_A.summary_df()

In [ ]:
NODE = {1: "n1", 2: "n2"}
rows = [(f"b_{r}_{i + 1}", -sA.loc[f"b_{r}_{i + 1}", "mean"], sA.loc[f"b_{r}_{i + 1}", "sd"],
         lme4_A.loc[f"item{it}:node{NODE[r]}", "estimate"], lme4_A.loc[f"item{it}:node{NODE[r]}", "se"])
        for r in (1, 2) for i, it in enumerate(ITEMS)]
cmp_A = compare(rows, "Model A, item locations (as -b)")
var_A = pd.DataFrame({"irtree_mean": sA.loc[["sigma2_1", "sigma2_2", "rho_1_2"], "mean"],
                      "irtree_sd":   sA.loc[["sigma2_1", "sigma2_2", "rho_1_2"], "sd"],
                      "glmer": lme4_A.loc[["var_id_noden1", "var_id_noden2", "cor_id_noden1_noden2"],
                                          "estimate"].to_numpy()})
var_A.round(3)

## Step 4 — Model B: explanatory
Random item locations with the package's default hierarchical prior; the item and person coefficients get N(0, 10). Compared: the 14 coefficients (glmer fixed effects), the trait variances and correlation, and the item variance of each node (glmer: `var_item_d1`, `var_item.1_d2`).

In [ ]:
lme4_B = pd.read_csv(RESULTS / "lme4_modelB.csv").set_index("term")
nodes_B = [irtree.LogitRaschNode(person_covariates=True, item_covariates=True) for _ in range(2)]
priors_B = irtree.build_default_priors(linear_tree, nodes_B)
priors_B.set("beta", VAGUE)
priors_B.set("kappa", VAGUE)
with timed("irtree Model B"):
    fit_B = irtree.irtree_fit(data_cov, linear_tree, nodes_B, priors=priors_B,
                              n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED, progress=False)
sB = fit_B.summary_df()

In [ ]:
def glmer_term(name):
    kind, r, cov = name.split("_", 2)
    return f"node{NODE[int(r)]}" if cov == "intercept" else f"node{NODE[int(r)]}:{cov}"

coef_names = [n for n in sB.index if n.startswith(("beta_", "kappa_"))]
rows = [(n, sB.loc[n, "mean"], sB.loc[n, "sd"], lme4_B.loc[glmer_term(n), "estimate"],
         lme4_B.loc[glmer_term(n), "se"]) for n in coef_names]
cmp_B = compare(rows, "Model B, coefficients")
display(cmp_B.round(3))
VC = {"sigma2_1": "var_id_noden1", "sigma2_2": "var_id_noden2", "rho_1_2": "cor_id_noden1_noden2",
      "b.variance_1": "var_item_d1", "b.variance_2": "var_item.1_d2"}
var_B = pd.DataFrame({"irtree_mean": sB.loc[list(VC), "mean"], "irtree_median": sB.loc[list(VC), "q50"],
                      "irtree_sd": sB.loc[list(VC), "sd"],
                      "glmer": lme4_B.loc[list(VC.values()), "estimate"].to_numpy()})
var_B.round(3)

## Step 5 — Prior sensitivity of the item variances
The item variance of a node is estimated from 24 locations. The default InvGamma(1, 1) prior has its mode at 0.5 and little mass near 0, so when the variance is small it pulls the posterior up. Refitting Model B under weaker InvGamma priors shows how much of the gap to glmer is the prior. This step refits Model B twice (about twice the runtime of Step 4).

In [ ]:
sens = {"InvGamma(1, 1) (default)": sB}
for a0, b0 in [(1.0, 0.1), (0.5, 0.05)]:
    pr = irtree.build_default_priors(linear_tree, nodes_B)
    pr.set("beta", VAGUE); pr.set("kappa", VAGUE)
    pr.set("b", irtree.NormalPrior(mean=0.0, variance=irtree.InvGammaPrior(alpha=a0, beta=b0)))
    with timed(f"irtree Model B, InvGamma({a0}, {b0})"):
        f = irtree.irtree_fit(data_cov, linear_tree, nodes_B, priors=pr, n_iter=N_ITER,
                              n_chains=N_CHAINS, seed=SEED, progress=False)
    sens[f"InvGamma({a0}, {b0})"] = f.summary_df()
keys = ["b.variance_1", "b.variance_2", "kappa_1_shout", "kappa_2_intercept"]
pd.concat({k: v.loc[keys, ["mean", "q50", "sd"]] for k, v in sens.items()}, axis=1).round(3)

# Part 2 — mirt
## Step 6 — Model M1: 2PL tree with correlated traits
Logistic 2PL nodes; the trait variances are fixed at 1 (irtree's default on dimensions whose nodes all have slopes, and mirt's convention), the correlation is estimated. For the comparison the locations get N(0, 10) and the slopes a wide TruncNormal(1, 4) on (0, ∞). mirt's intercept is d = −a·b, computed draw by draw.

In [ ]:
mirt_M1 = pd.read_csv(RESULTS / "mirt_M1_items.csv")
mirt_cor = pd.read_csv(RESULTS / "mirt_M1_cor.csv")
nodes_M1 = [irtree.LogitTwoPLNode(), irtree.LogitTwoPLNode()]
priors_M1 = irtree.build_default_priors(linear_tree, nodes_M1)
priors_M1.set("b", VAGUE)
priors_M1.set("a", irtree.TruncNormalPrior(mean=1.0, variance=4.0, lower=0.0))
with timed("irtree Model M1"):
    fit_M1 = irtree.irtree_fit(data_plain, linear_tree, nodes_M1, priors=priors_M1,
                               n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED, progress=False)

In [ ]:
rows_a, rows_d = [], []
for row in mirt_M1.itertuples():
    a = draws(fit_M1, f"a_{row.node}_{row.item_index}")
    d = -a * draws(fit_M1, f"b_{row.node}_{row.item_index}")
    rows_a.append((f"a_{row.node}_{row.item_index}", a.mean(), a.std(), row.a, row.a_se))
    rows_d.append((f"d_{row.node}_{row.item_index}", d.mean(), d.std(), row.d, row.d_se))
cmp_M1a = compare(rows_a, "M1 slopes a")
cmp_M1d = compare(rows_d, "M1 intercepts d = -a*b")
rho = draws(fit_M1, "rho_1_2")
print(f"correlation: irtree {rho.mean():.3f} (sd {rho.std():.3f}),  "
      f"mirt {mirt_cor['estimate'][0]:.3f} (se {mirt_cor['se'][0]:.3f})")

## Step 7 — Model M2: one-node generalized partial credit model
A tree with one node and three outcomes is an ordinary GPCM. The step parameters get N(0, 10), the slopes TruncNormal(1, 4).

In [ ]:
mirt_M2 = pd.read_csv(RESULTS / "mirt_M2_items.csv")
one_node = irtree.IRTreeSpec(mapping=np.array([[0], [1], [2]]))
data_one = irtree.load_data(table[["id", "item", "response"]], spec=one_node, person_id="id",
                            item_id="item", response_col="response", categories_start=0)
nodes_M2 = [irtree.GPCMNode(n_outcomes=3)]
priors_M2 = irtree.build_default_priors(one_node, nodes_M2)
priors_M2.set("b", VAGUE)
priors_M2.set("a", irtree.TruncNormalPrior(mean=1.0, variance=4.0, lower=0.0))
with timed("irtree Model M2"):
    fit_M2 = irtree.irtree_fit(data_one, one_node, nodes_M2, priors=priors_M2,
                               n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED, progress=False)
sM2 = fit_M2.summary_df()
rows = []
for row in mirt_M2.itertuples():
    j = row.item_index
    rows += [(f"a_1_{j}", sM2.loc[f"a_1_{j}", "mean"], sM2.loc[f"a_1_{j}", "sd"], row.a, row.a_se),
             (f"b_1_{j}_thresh_1", sM2.loc[f"b_1_{j}_thresh_1", "mean"], sM2.loc[f"b_1_{j}_thresh_1", "sd"], row.b1, row.b1_se),
             (f"b_1_{j}_thresh_2", sM2.loc[f"b_1_{j}_thresh_2", "mean"], sM2.loc[f"b_1_{j}_thresh_2", "sd"], row.b2, row.b2_se)]
cmp_M2 = compare(rows, "M2 GPCM slopes and steps")

## Step 8 — Figure and tables for the paper
One panel per comparison; points on the diagonal agree. Error bars are ±1 posterior SD.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
scatter(cmp_A, "lme4, Model A: locations", axes[0])
scatter(cmp_B, "lme4, Model B: coefficients", axes[1])
scatter(pd.concat([cmp_M1a, cmp_M1d]), "mirt, M1: a and d", axes[2])
scatter(cmp_M2, "mirt, M2: GPCM a and steps", axes[3])
fig.tight_layout()
fig.savefig(RESULTS / "agreement.png", dpi=200)
for name, df in {"lme4_A": cmp_A, "lme4_B": cmp_B, "mirt_M1_a": cmp_M1a,
                 "mirt_M1_d": cmp_M1d, "mirt_M2": cmp_M2}.items():
    df.to_csv(RESULTS / f"comparison_{name}.csv", index=False)
var_A.to_csv(RESULTS / "comparison_lme4_A_variances.csv")
var_B.to_csv(RESULTS / "comparison_lme4_B_variances.csv")

## Runtime

In [ ]:
rt = pd.DataFrame(RUNTIME)
rt.loc[len(rt)] = {"step": "TOTAL", "seconds": rt["seconds"].sum()}
rt.to_csv(RESULTS / "runtime_72.csv", index=False)
rt